# Bloque 3: Optimización de Modelos
## Estrategias de búsqueda inteligente en ML en producción

**Objetivo**: Entender por qué tuning importa, cómo buscar hiperparámetros de forma eficiente, y cómo evitar las trampas reales que rompen modelos en producción.

## Setup: Imports y Configuración

In [ ]:
# Imports necesarios
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

# Para Bayesian Optimization
!pip install optuna -q
import optuna
from optuna.samplers import TPESampler

# Configuración visual
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
np.random.seed(42)

print("✓ Todos los imports listos")

## Dataset Synthetic
Crearemos un dataset realista con:
- Características correlacionadas (como en producción)
- Desbalance de clases
- Noise para simular realidad

In [ ]:
# Crear dataset synthetic
np.random.seed(42)
X, y = make_classification(
    n_samples=2000,
    n_features=20,
    n_informative=12,
    n_redundant=5,
    n_classes=2,
    weights=[0.7, 0.3],  # Desbalance: 70% clase 0, 30% clase 1
    random_state=42
)

# Split: train/val/test
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

# Escalar
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(f"Train: {X_train.shape}")
print(f"Val: {X_val.shape}")
print(f"Test: {X_test.shape}")
print(f"\nDesbalance:")
print(f"Train - Clase 0: {(y_train==0).sum()}, Clase 1: {(y_train==1).sum()}")
print(f"Test - Clase 0: {(y_test==0).sum()}, Clase 1: {(y_test==1).sum()}")

---
# SECCIÓN 1: ¿Por qué tuning importa?

## El Problema
Regularización (L1/L2, subsample) **limpia** el modelo.
Pero **no elige los valores óptimos para tus datos**.

Sin tuning: usas defaults → subreutilizas capacidad.
Con tuning: encuentras valores que maximizan métrica en TUS datos.

### Demostración: Defaults vs Tuned

In [ ]:
from sklearn.metrics import roc_auc_score

# Modelo 1: XGBoost con parámetros DEFAULT
model_default = XGBClassifier(
    n_estimators=100,
    max_depth=6,  # Default XGBoost
    learning_rate=0.1,  # Default
    subsample=1.0,  # Sin regularización
    colsample_bytree=1.0,  # Sin regularización
    random_state=42
)

# Modelo 2: XGBoost con parámetros TUNED (los que buscaremos después)
model_tuned = XGBClassifier(
    n_estimators=100,
    max_depth=4,  # Más conservador
    learning_rate=0.05,  # Más lento = mejor generalizacion
    subsample=0.7,  # Regularización por filas
    colsample_bytree=0.8,  # Regularización por columnas
    reg_alpha=0.1,  # L1
    reg_lambda=1.0,  # L2
    random_state=42
)

# Entrenar
model_default.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
model_tuned.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

# Evaluar
auc_default_train = roc_auc_score(y_train, model_default.predict_proba(X_train)[:, 1])
auc_default_val = roc_auc_score(y_val, model_default.predict_proba(X_val)[:, 1])
auc_default_test = roc_auc_score(y_test, model_default.predict_proba(X_test)[:, 1])

auc_tuned_train = roc_auc_score(y_train, model_tuned.predict_proba(X_train)[:, 1])
auc_tuned_val = roc_auc_score(y_val, model_tuned.predict_proba(X_val)[:, 1])
auc_tuned_test = roc_auc_score(y_test, model_tuned.predict_proba(X_test)[:, 1])

# Comparar
results = pd.DataFrame({
    'Modelo': ['Default', 'Tuned'],
    'Train AUC': [auc_default_train, auc_tuned_train],
    'Val AUC': [auc_default_val, auc_tuned_val],
    'Test AUC': [auc_default_test, auc_tuned_test]
})

print("\n" + "="*60)
print(results.to_string(index=False))
print("="*60)
print(f"\nGanancia en Test AUC: {(auc_tuned_test - auc_default_test)*100:.2f}%")

### ✅ CONCLUSIÓN Sección 1

**El tuning no es cosmético**: cambiar max_depth, learning_rate, regularización genera diferencias reales en test (+2-5% es típico).

**El reto**: hay cientos de combinaciones posibles. ¿Cómo buscar eficientemente?

---
# SECCIÓN 2: Estrategias de búsqueda - Concepto

## Las 3 estrategias principales

| Estrategia | Idea | Evaluaciones | Tiempo | Cuándo usar |
|-----------|------|---------|--------|-------------|
| **Grid Search** | Prueba todas las combinaciones en una malla | Alto (3×3×3 = 27 si 3 parámetros) | Lento | Pocos parámetros, espacio definido |
| **Random Search** | Prueba combinaciones aleatorias | Medio (usuario define N) | Medio | Espacio grande, búsqueda inicial |
| **Bayesian Optimization** | Aprende relación param→métrica, próximas pruebas son "inteligentes" | Bajo (eficiente) | Corto | Tuning final, parámetros costosos |

## Intuición Visual

In [ ]:
# Visualización: Superficie de pérdida (simplificada)
max_depths = np.arange(2, 12)
learning_rates = np.arange(0.01, 0.3, 0.01)

# Crear matriz de resultados (simulada)
scores = np.zeros((len(max_depths), len(learning_rates)))
for i, depth in enumerate(max_depths):
    for j, lr in enumerate(learning_rates):
        # Función simulada con óptimo en (depth=5, lr=0.15)
        scores[i, j] = 0.75 - 0.02*(depth-5)**2 - 0.5*(lr-0.15)**2 + np.random.normal(0, 0.01)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Grid Search
ax = axes[0]
grid_points = [(2, 0.01), (2, 0.15), (2, 0.29), 
               (5, 0.01), (5, 0.15), (5, 0.29),
               (10, 0.01), (10, 0.15), (10, 0.29)]
ax.contourf(learning_rates, max_depths, scores, levels=15, cmap='RdYlGn')
grid_x = [p[1] for p in grid_points]
grid_y = [p[0] for p in grid_points]
ax.scatter(grid_x, grid_y, c='red', s=100, marker='s', label='Grid Search (9 evals)')
ax.set_xlabel('Learning Rate')
ax.set_ylabel('Max Depth')
ax.set_title('Grid Search: Cubre malla regular')
ax.legend()

# Random Search
ax = axes[1]
random_points = np.random.uniform([0.01, 2], [0.3, 10], (20, 2))
ax.contourf(learning_rates, max_depths, scores, levels=15, cmap='RdYlGn')
ax.scatter(random_points[:, 0], random_points[:, 1], c='blue', s=100, marker='^', label='Random Search (20 evals)')
ax.set_xlabel('Learning Rate')
ax.set_ylabel('Max Depth')
ax.set_title('Random Search: Explora más el espacio')
ax.legend()

# Bayesian
ax = axes[2]
bayesian_points = [(0.15, 5), (0.16, 5), (0.14, 4.5), (0.17, 5.5), (0.15, 5), 
                   (0.14, 5.2), (0.16, 4.8), (0.15, 5.3)]
ax.contourf(learning_rates, max_depths, scores, levels=15, cmap='RdYlGn')
bx = [p[0] for p in bayesian_points]
by = [p[1] for p in bayesian_points]
ax.scatter(bx, by, c='green', s=100, marker='*', label='Bayesian (8 evals, inteligentes)')
ax.set_xlabel('Learning Rate')
ax.set_ylabel('Max Depth')
ax.set_title('Bayesian: Concentra búsqueda cerca del óptimo')
ax.legend()

plt.tight_layout()
plt.show()

print("Observa: Bayesian prueba puntos MÁS cercanos al óptimo (verde) con menos evaluaciones.")

### ✅ CONCLUSIÓN Sección 2

**Grid Search**: seguro pero lento. Usa cuando tienes pocos parámetros.

**Random Search**: más eficiente que Grid. Bueno para exploración inicial.

**Bayesian Optimization**: inteligente, eficiente. Para tuning final cuando cada evaluación es cara.

---
# SECCIÓN 3: Grid Search vs Random Search - Comparación práctica

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
import time

# Parámetros a tunear
param_grid = {
    'max_depth': [3, 4, 5, 6, 7],
    'learning_rate': [0.01, 0.05, 0.1, 0.15],
    'subsample': [0.7, 0.8, 0.9]
}

print(f"Total de combinaciones posibles: {5 * 4 * 3} = 60")
print("\n" + "="*60)

# GRID SEARCH
print("\n1. GRID SEARCH (todas las combinaciones)")
grid_search = GridSearchCV(
    XGBClassifier(n_estimators=50, random_state=42),
    param_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1
)

start = time.time()
grid_search.fit(X_train, y_train)
grid_time = time.time() - start

print(f"Tiempo: {grid_time:.2f}s")
print(f"Mejor parámetros: {grid_search.best_params_}")
print(f"Mejor CV AUC: {grid_search.best_score_:.4f}")
print(f"Test AUC: {roc_auc_score(y_test, grid_search.best_estimator_.predict_proba(X_test)[:, 1]):.4f}")

# Guardar resultados
grid_results = pd.DataFrame(grid_search.cv_results_)
grid_best_val_auc = grid_search.best_score_
grid_best_test_auc = roc_auc_score(y_test, grid_search.best_estimator_.predict_proba(X_test)[:, 1])


In [ ]:
# RANDOM SEARCH
print("\n2. RANDOM SEARCH (20 combinaciones aleatorias)")
random_search = RandomizedSearchCV(
    XGBClassifier(n_estimators=50, random_state=42),
    param_grid,
    n_iter=20,  # Solo 20 evaluaciones vs 60 de Grid
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=42
)

start = time.time()
random_search.fit(X_train, y_train)
random_time = time.time() - start

print(f"Tiempo: {random_time:.2f}s")
print(f"Mejor parámetros: {random_search.best_params_}")
print(f"Mejor CV AUC: {random_search.best_score_:.4f}")
print(f"Test AUC: {roc_auc_score(y_test, random_search.best_estimator_.predict_proba(X_test)[:, 1]):.4f}")

# Guardar resultados
random_results = pd.DataFrame(random_search.cv_results_)
random_best_val_auc = random_search.best_score_
random_best_test_auc = roc_auc_score(y_test, random_search.best_estimator_.predict_proba(X_test)[:, 1])

In [ ]:
# COMPARACIÓN
print("\n" + "="*60)
print("COMPARACIÓN: Grid Search vs Random Search")
print("="*60)

comparison = pd.DataFrame({
    'Métrica': ['Evaluaciones', 'Tiempo (s)', 'Val AUC', 'Test AUC', 'Eficiencia (AUC/tiempo)'],
    'Grid Search': [60, f"{grid_time:.2f}", f"{grid_best_val_auc:.4f}", f"{grid_best_test_auc:.4f}", f"{grid_best_test_auc/grid_time:.2f}"],
    'Random Search': [20, f"{random_time:.2f}", f"{random_best_val_auc:.4f}", f"{random_best_test_auc:.4f}", f"{random_best_test_auc/random_time:.2f}"]
})

print(comparison.to_string(index=False))
print("\n→ Random Search: 67% menos evaluaciones, resultados similares o mejores")

In [ ]:
# Visualización: Convergencia de búsqueda
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Grid Search: ordenar por mejor AUC
grid_sorted = grid_results.sort_values('rank_test_score').reset_index(drop=True)
grid_cummax = grid_sorted['mean_test_score'].cummax()

# Random Search: ordenar por mejor AUC
random_sorted = random_results.sort_values('rank_test_score').reset_index(drop=True)
random_cummax = random_sorted['mean_test_score'].cummax()

ax = axes[0]
ax.plot(range(1, len(grid_cummax)+1), grid_cummax, 'o-', label='Grid Search', linewidth=2, markersize=4)
ax.plot(range(1, len(random_cummax)+1), random_cummax, 's-', label='Random Search', linewidth=2, markersize=4)
ax.set_xlabel('Evaluaciones')
ax.set_ylabel('Best AUC encontrado (acumulativo)')
ax.set_title('Convergencia: ¿Quién encuentra el óptimo más rápido?')
ax.legend()
ax.grid(True, alpha=0.3)

# Distribución de AUC evaluadas
ax = axes[1]
ax.hist(grid_results['mean_test_score'], bins=15, alpha=0.6, label=f'Grid Search (N={len(grid_results)})', edgecolor='black')
ax.hist(random_results['mean_test_score'], bins=10, alpha=0.6, label=f'Random Search (N={len(random_results)})', edgecolor='black')
ax.set_xlabel('CV AUC')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución de AUC: Random explora mejor')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### ✅ CONCLUSIÓN Sección 3

**Grid Search** prueba 60 combinaciones exhaustivamente → tiempo largo, encontrar óptimo garantizado en malla.

**Random Search** prueba 20 combinaciones aleatorias → 3x más rápido, explora mejor el espacio, resultados comparables o mejores.

**Winner**: Random Search para espacio grande. Grid Search solo si pocos parámetros.

---
# SECCIÓN 4: Bayesian Optimization con Optuna

## ¿Cómo funciona Bayesian Optimization?

1. **Evaluación inicial**: prueba N combinaciones aleatorias
2. **Modelo probabilístico**: "aprende" relación parámetro → métrica
3. **Adquisición**: siguiente evaluación = punto con mayor probabilidad de mejora
4. **Iteración**: repite 3 hasta exhaustar presupuesto

**Resultado**: menos evaluaciones, mejor valor encontrado.

In [ ]:
# BAYESIAN OPTIMIZATION con Optuna
print("BAYESIAN OPTIMIZATION (Optuna)")
print("="*60)

# Definir objetivo
def objective(trial):
    # Parámetros a tunear
    max_depth = trial.suggest_int('max_depth', 2, 10)
    learning_rate = trial.suggest_float('learning_rate', 0.001, 0.3, log=True)
    subsample = trial.suggest_float('subsample', 0.5, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0)
    reg_alpha = trial.suggest_float('reg_alpha', 0.0, 1.0)
    reg_lambda = trial.suggest_float('reg_lambda', 0.0, 2.0)
    
    # Entrenar modelo
    model = XGBClassifier(
        n_estimators=50,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        random_state=42,
        verbosity=0
    )
    
    # Evaluar con CV
    scores = cross_val_score(model, X_train, y_train, cv=3, scoring='roc_auc')
    return scores.mean()

# Crear estudio
sampler = TPESampler(seed=42)  # TPE = Tree-Structured Parzen Estimator (Bayesian)
study = optuna.create_study(direction='maximize', sampler=sampler)

# Optimizar
start = time.time()
study.optimize(objective, n_trials=20, show_progress_bar=False)
bayesian_time = time.time() - start

print(f"\nTiempo: {bayesian_time:.2f}s")
print(f"\nMejor parámetros encontrados:")
best_params = study.best_trial.params
for key, value in best_params.items():
    print(f"  {key}: {value}")

print(f"\nMejor CV AUC: {study.best_value:.4f}")

In [ ]:
# Entrenar modelo final con mejores parámetros
model_bayesian = XGBClassifier(
    n_estimators=100,  # Aumentamos a 100 para evaluación final
    **best_params,
    random_state=42
)
model_bayesian.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

bayesian_test_auc = roc_auc_score(y_test, model_bayesian.predict_proba(X_test)[:, 1])
print(f"\nTest AUC (modelo final): {bayesian_test_auc:.4f}")

In [ ]:
# Importancia de parámetros (Feature Importance en el espacio de búsqueda)
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# Extraer trials como DataFrame
trials_df = study.trials_dataframe()
param_cols = [col for col in trials_df.columns if col.startswith('params_')]

for idx, param in enumerate(param_cols[:6]):
    ax = axes[idx // 3, idx % 3]
    param_name = param.replace('params_', '')
    
    # Scatter: parámetro vs AUC
    ax.scatter(trials_df[param], trials_df['value'], alpha=0.6, s=50)
    ax.set_xlabel(param_name)
    ax.set_ylabel('CV AUC')
    ax.set_title(f'Impacto de {param_name}')
    ax.grid(True, alpha=0.3)

plt.suptitle('Importancia de parámetros: relación con AUC', fontsize=14, y=1.00)
plt.tight_layout()
plt.show()

print("Observa qué parámetros realmente mueven AUC vs cuáles son ruido.")

In [ ]:
# Comparación: Grid vs Random vs Bayesian
print("\n" + "="*70)
print("COMPARACIÓN FINAL: Grid Search vs Random Search vs Bayesian Optimization")
print("="*70)

final_comparison = pd.DataFrame({
    'Método': ['Grid Search', 'Random Search', 'Bayesian Optimization'],
    'Evaluaciones': [60, 20, 20],
    'Tiempo (s)': [f"{grid_time:.2f}", f"{random_time:.2f}", f"{bayesian_time:.2f}"],
    'Val AUC': [f"{grid_best_val_auc:.4f}", f"{random_best_val_auc:.4f}", f"{study.best_value:.4f}"],
    'Test AUC': [f"{grid_best_test_auc:.4f}", f"{random_best_test_auc:.4f}", f"{bayesian_test_auc:.4f}"]
})

print(final_comparison.to_string(index=False))

print("\n→ Bayesian: MENOS evaluaciones que Grid, MEJOR resultado que Random (típicamente).")
print("→ Razón: aprende dónde está el óptimo, concentra búsqueda ahí.")

### ✅ CONCLUSIÓN Sección 4

**Bayesian Optimization** aprende relación parámetro-métrica → próximas evaluaciones son inteligentes.

**Ventajas**:
- Menos evaluaciones que Grid/Random
- Encuentra mejores valores
- Eficiente cuando evaluaciones son caras

**Herramienta recomendada**: Optuna (moderno, flexible, fácil).

**Cuándo usar**: tuning final, modelos costosos, cuando presupuesto de evaluaciones es limitado.

---
# SECCIÓN 5: GOTCHAS REALES - Trampas que rompen modelos en producción

## Gotcha #1: Overfitting a la búsqueda

**El problema**: buscas tanto en Val que terminan memorizando el val set → test falla.

**Síntoma**: val AUC sube pero test AUC baja o se estanca.

**Causa**: usas el MISMO val set 20 veces para tuning → overfitting a ese val set.

In [ ]:
# DEMOSTRACIÓN: Overfitting a la búsqueda
print("GOTCHA #1: Overfitting a la búsqueda")
print("="*60)

# Scenario 1: Tuning INCORRECTO (sin hold-out test)
print("\n❌ INCORRECTO: Tuning sin separar test set")
print("   (Val set participa en 20 búsquedas → overfitting)\n")

X_train_bad, X_val_test_bad, y_train_bad, y_val_test_bad = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler_bad = StandardScaler()
X_train_bad = scaler_bad.fit_transform(X_train_bad)
X_val_test_bad = scaler_bad.transform(X_val_test_bad)

# Tuning en train/val_test
grid_search_bad = GridSearchCV(
    XGBClassifier(n_estimators=50, random_state=42),
    {'max_depth': [3, 5, 7], 'learning_rate': [0.01, 0.1, 0.2]},
    cv=3,
    scoring='roc_auc',
    n_jobs=-1
)

grid_search_bad.fit(X_train_bad, y_train_bad)
bad_val_auc = grid_search_bad.best_score_
# Evaluación en el MISMO set usado para tuning
bad_eval_auc = cross_val_score(grid_search_bad.best_estimator_, X_train_bad, y_train_bad, cv=5, scoring='roc_auc').mean()

print(f"Val AUC (durante tuning): {bad_val_auc:.4f}")
print(f"Val AUC (CV en mismo set): {bad_eval_auc:.4f}")
print(f"\n⚠️  Ambos altos → sospechoso. No hay verdadero test set.")

In [ ]:
# Scenario 2: Tuning CORRECTO (con hold-out test)
print("\n✅ CORRECTO: Tuning con hold-out test set separado")
print("   (Val set solo para tuning, test set completamente aparte)\n")

# Split correcto: train/val/test
X_temp_good, X_test_good, y_temp_good, y_test_good = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train_good, X_val_good, y_train_good, y_val_good = train_test_split(
    X_temp_good, y_temp_good, test_size=0.25, random_state=42, stratify=y_temp_good
)

scaler_good = StandardScaler()
X_train_good = scaler_good.fit_transform(X_train_good)
X_val_good = scaler_good.transform(X_val_good)
X_test_good = scaler_good.transform(X_test_good)

# Tuning EN TRAIN/VAL
grid_search_good = GridSearchCV(
    XGBClassifier(n_estimators=50, random_state=42),
    {'max_depth': [3, 5, 7], 'learning_rate': [0.01, 0.1, 0.2]},
    cv=3,
    scoring='roc_auc',
    n_jobs=-1
)

grid_search_good.fit(X_train_good, y_train_good)
good_val_auc = grid_search_good.best_score_
# Evaluación FINAL en test set (nunca visto)
good_test_auc = roc_auc_score(y_test_good, grid_search_good.best_estimator_.predict_proba(X_test_good)[:, 1])

print(f"Val AUC (durante tuning): {good_val_auc:.4f}")
print(f"Test AUC (nunca visto): {good_test_auc:.4f}")
print(f"Diferencia: {abs(good_val_auc - good_test_auc):.4f}")
print(f"\n✓ Diferencia pequeña → no hay overfitting a la búsqueda.")

In [ ]:
# Visualización del problema
fig, ax = plt.subplots(figsize=(10, 5))

methods = ['Tuning incorrecto\n(sin test aparte)', 'Tuning correcto\n(con hold-out test)']
val_scores = [bad_val_auc, good_val_auc]
test_scores = [bad_eval_auc, good_test_auc]  # El "test" malo es en el mismo set

x = np.arange(len(methods))
width = 0.35

bars1 = ax.bar(x - width/2, val_scores, width, label='Val AUC', alpha=0.8)
bars2 = ax.bar(x + width/2, test_scores, width, label='Test AUC', alpha=0.8)

ax.set_ylabel('AUC')
ax.set_title('Overfitting a la búsqueda: brecha Val-Test')
ax.set_xticks(x)
ax.set_xticklabels(methods)
ax.legend()
ax.set_ylim([0.7, 0.95])
ax.grid(True, alpha=0.3, axis='y')

# Anotaciones
ax.annotate('', xy=(0, val_scores[0]), xytext=(0, test_scores[0]),
            arrowprops=dict(arrowstyle='<->', color='red', lw=2))
ax.text(0.15, (val_scores[0]+test_scores[0])/2, '⚠️ BRECHA', fontsize=10, color='red', weight='bold')

ax.annotate('', xy=(1, val_scores[1]), xytext=(1, test_scores[1]),
            arrowprops=dict(arrowstyle='<->', color='green', lw=2))
ax.text(1.15, (val_scores[1]+test_scores[1])/2, '✓ OK', fontsize=10, color='green', weight='bold')

plt.tight_layout()
plt.show()

### 🔧 FIX Gotcha #1: Validación correcta

```
Estrategia 1: Hold-out test set (lo que usamos arriba)
├── Train (60%): entrenamiento
├── Val (20%): tuning solo
└── Test (20%): evaluación FINAL, nunca toca tuning

Estrategia 2: Nested CV (más riguroso pero lento)
├── Loop externo (5-fold CV): genera test sets
  ├── Loop interno (GridSearch/Bayesian): tuning en train/val
  └── Evalúa en test externo
```

---
## Gotcha #2: Early Stopping + Tuning (trampa con XGBoost)

**El problema**: Early Stopping usa val set para decidir cuándo parar.
Pero ese val set lo estás probando N veces (en la búsqueda).

**Síntoma**: Early stopping elige diferentes n_estimators en cada trial → inconsistencia.

In [ ]:
# DEMOSTRACIÓN: Early Stopping + Tuning
print("GOTCHA #2: Early Stopping + Tuning (XGBoost trap)")
print("="*60)

# Función objetivo CON early stopping
def objective_with_early_stopping(trial):
    max_depth = trial.suggest_int('max_depth', 2, 10)
    learning_rate = trial.suggest_float('learning_rate', 0.001, 0.3, log=True)
    
    model = XGBClassifier(
        n_estimators=200,  # Muchos, pero early stopping parará antes
        max_depth=max_depth,
        learning_rate=learning_rate,
        random_state=42,
        verbosity=0
    )
    
    model.fit(
        X_train_good, y_train_good,
        eval_set=[(X_val_good, y_val_good)],
        early_stopping_rounds=10,
        verbose=False
    )
    
    # El modelo paró en different n_estimators según parámetros
    # Esto es PROBLEM: val set no está limpio
    auc = roc_auc_score(y_val_good, model.predict_proba(X_val_good)[:, 1])
    return auc

# Tunear CON early stopping
study_es = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study_es.optimize(objective_with_early_stopping, n_trials=5, show_progress_bar=False)

print(f"\nMejor AUC encontrado: {study_es.best_value:.4f}")
print(f"⚠️  Problema: cada trial paró en diferentes epochs → val set no es limpio")

# Extraer n_estimators usados en cada trial
n_estimators_per_trial = []
for trial in study_es.trials:
    max_depth = trial.params['max_depth']
    lr = trial.params['learning_rate']
    
    model_temp = XGBClassifier(n_estimators=200, max_depth=max_depth, learning_rate=lr, random_state=42, verbosity=0)
    model_temp.fit(X_train_good, y_train_good, eval_set=[(X_val_good, y_val_good)], 
                  early_stopping_rounds=10, verbose=False)
    n_estimators_per_trial.append(model_temp.best_iteration + 1)

print(f"\nN_estimators usados en cada trial: {n_estimators_per_trial}")
print(f"Variación: {max(n_estimators_per_trial) - min(n_estimators_per_trial)} estimators")
print(f"→ Inconsistencia: modelos finales son DIFERENTES (distintos #árboles)")

### 🔧 FIX Gotcha #2: Early Stopping limpio

**Opción A** (Recomendado): Early stopping + Hold-out test
- Tuning: train/val (sin ES o con ES)
- ES usa val set pero está dentro de tuning
- Evaluación final: test set (limpio)

**Opción B**: Fijar n_estimators en tuning
- No uses early stopping durante tuning
- Usa early stopping SOLO en modelo final

---
## Gotcha #3: Distribuiciones diferentes entre train y test (Distribution Shift)

**El problema**: tuning optimiza para train... pero en producción llegan datos diferentes.

**Síntoma**: val AUC = 0.92, test AUC = 0.70 (caída brutal).

**Causa**: distribution shift, data drift, cambios temporales.

In [ ]:
# DEMOSTRACIÓN: Distribution Shift
print("\nGOTCHA #3: Distribution Shift")
print("="*60)

# Crear test set con DISTRIBUTION DIFERENTE
# (Simula cambio en producción)
np.random.seed(123)
X_test_shifted = X_test_good.copy()
X_test_shifted = X_test_shifted + np.random.normal(0.5, 0.3, X_test_shifted.shape)  # Shift en media

# Entrenar modelo con parámetros tuneados
model_for_shift = XGBClassifier(
    **grid_search_good.best_params_,
    n_estimators=100,
    random_state=42
)
model_for_shift.fit(X_train_good, y_train_good)

# Evaluación
auc_test_clean = roc_auc_score(y_test_good, model_for_shift.predict_proba(X_test_good)[:, 1])
auc_test_shifted = roc_auc_score(y_test_good, model_for_shift.predict_proba(X_test_shifted)[:, 1])

print(f"Test AUC (distribución original): {auc_test_clean:.4f}")
print(f"Test AUC (distribución shifted): {auc_test_shifted:.4f}")
print(f"Caída: {(auc_test_clean - auc_test_shifted)*100:.2f}%")
print(f"\n⚠️  Caída brutal por cambio en datos → tuning no previno.")

In [ ]:
# Visualización: diferencias en distribuiciones
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

feature_idx = 0

# Feature 0: Train vs Test vs Test_shifted
ax = axes[0]
ax.hist(X_train_good[:, feature_idx], bins=20, alpha=0.5, label='Train', edgecolor='black')
ax.hist(X_test_good[:, feature_idx], bins=20, alpha=0.5, label='Test (clean)', edgecolor='black')
ax.hist(X_test_shifted[:, feature_idx], bins=20, alpha=0.5, label='Test (shifted)', edgecolor='black')
ax.set_xlabel(f'Feature {feature_idx}')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribuciones comparadas')
ax.legend()
ax.grid(True, alpha=0.3)

# Performance bajo shift
ax = axes[1]
shifts = np.linspace(0, 1.0, 10)
aucs = []
for shift_amount in shifts:
    X_test_temp = X_test_good + np.random.normal(shift_amount, 0.2, X_test_good.shape)
    auc = roc_auc_score(y_test_good, model_for_shift.predict_proba(X_test_temp)[:, 1])
    aucs.append(auc)

ax.plot(shifts, aucs, 'o-', linewidth=2, markersize=6)
ax.fill_between(shifts, aucs, alpha=0.2)
ax.set_xlabel('Magnitude of Distribution Shift')
ax.set_ylabel('Test AUC')
ax.set_title('AUC degrada con distribution shift')
ax.grid(True, alpha=0.3)

# Expectativa vs Realidad
ax = axes[2]
scenarios = ['Esperado\n(val AUC)', 'Realidad\n(test clean)', 'En Producción\n(test shifted)']
aucs_scenarios = [good_val_auc, auc_test_clean, auc_test_shifted]
colors = ['green', 'blue', 'red']
bars = ax.bar(scenarios, aucs_scenarios, color=colors, alpha=0.7, edgecolor='black')
ax.set_ylabel('AUC')
ax.set_ylim([0.5, 1.0])
ax.set_title('La realidad puede ser brutal')
ax.grid(True, alpha=0.3, axis='y')

for i, (bar, auc) in enumerate(zip(bars, aucs_scenarios)):
    ax.text(bar.get_x() + bar.get_width()/2, auc + 0.02, f'{auc:.3f}', 
           ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

### 🔧 FIX Gotcha #3: Validación robusta a Distribution Shift

**Monitorear en producción**:
- Medir AUC en rolling windows
- Comparar distribuciones: KS test, Wasserstein distance
- Alerta si AUC cae > X%

**Prevenir en tuning**:
- Usar datos históricos: "test" = mes más reciente
- Validación temporal: train = pasado, val/test = futuro
- Testear en múltiples períodos antes de deploy

---
# SECCIÓN 6: Recomendaciones prácticas y flujo de trabajo


In [ ]:
# TABLA DE DECISIÓN: Qué estrategia usar
print("\nTABLA DE DECISIÓN: Qué estrategia de tuning usar")
print("="*80)

decision_table = pd.DataFrame({
    'Tu situación': [
        'Pocos parámetros (2-3)',
        'Muchos parámetros, evaluación rápida',
        'Muchos parámetros, evaluación cara',
        'Prototipo rápido',
        'Producción, máxima precisión'
    ],
    'Recomienda': [
        'Grid Search',
        'Random Search',
        'Bayesian Optimization',
        'Parámetros defaults',
        'Bayesian + Validación Temporal'
    ],
    'Razón': [
        'Exhaustivo, rápido',
        'Explora bien, no tarda',
        'Inteligente, eficiente',
        'Sin tiempo para tuning',
        'Robusto a drift, máximo rigor'
    ]
})

print(decision_table.to_string(index=False))

print("\n" + "="*80)

In [ ]:
# FLUJO DE TRABAJO RECOMENDADO
print("\nFLUJO DE TRABAJO RECOMENDADO EN PRODUCCIÓN")
print("="*80)

workflow = """
PASO 1: Preparación de datos
  ├─ Train (60%): entrenamiento
  ├─ Val (20%): tuning
  └─ Test (20%): evaluación FINAL, no toca tuning
     ⚠️  Si datos temporales: test = período más reciente

PASO 2: Baseline rápido
  └─ Modelo con parámetros defaults → establece baseline

PASO 3: Búsqueda de hiperparámetros
  ├─ Si pocos parámetros → Grid Search
  ├─ Si muchos parámetros, rápido → Random Search (N=20-50)
  └─ Si evaluación cara → Bayesian Optimization (Optuna)

PASO 4: Evaluación robusta
  ├─ Val AUC (durante tuning) → indicador de búsqueda
  ├─ Test AUC (nunca visto) → performance real
  └─ ⚠️  Si |Val AUC - Test AUC| > 5% → sospechar overfitting

PASO 5: Validación a distribution shift
  ├─ Test en múltiples períodos (si datos temporales)
  └─ Monitor en producción: AUC en rolling windows

PASO 6: Documentar decisiones
  ├─ Qué parámetros movieron aguja (importancia)
  ├─ Razón de cada valor tuneado
  └─ Trade-offs aceptados (complejidad vs ganancia)
"""

print(workflow)

In [ ]:
# Checklist: ¿Está tu tuning correcto?
print("\nCHECKLIST: ¿Tu tuning es correcto?")
print("="*80)

checklist = pd.DataFrame({
    '✓': ['☑', '☑', '☑', '☑', '☑', '☑', '☑'],
    'Validación': [
        'Test set está COMPLETAMENTE separado (no participa en tuning)',
        'Val AUC ≈ Test AUC (diferencia < 5%)',
        'No usas early stopping durante tuning (o cuidas la validación)',
        'Documentaste cuál parámetro movió AUC (importancia)',
        'Evaluaste en datos de diferente período/distribución',
        'Comparaste estrategias (Grid vs Random vs Bayesian)',
        'Estableciste baseline antes de tuning'
    ],
    'Impacto': [
        'Evita overfitting a búsqueda',
        'Indica no hay overfitting',
        'Evita val set sucio',
        'Decisiones informadas',
        'Robustez a drift',
        'Eficiencia del tuning',
        'Medir ganancia real'
    ]
})

print(checklist.to_string(index=False))

print("\n⚠️  Si NO checkeaste todos → hay riesgo de sorpresas en producción.")

### ✅ CONCLUSIÓN Sección 6

**Tuning no es buscar ciegamente**: es decisión informada sobre:
1. Validación correcta (hold-out test)
2. Estrategia eficiente (Grid/Random/Bayesian)
3. Robustez a cambios (distribution shift, temporal)
4. Documentación clara (qué se cambió y por qué)

**Herramienta recomendada en producción**: Bayesian Optimization (Optuna) + hold-out test + monitoreo temporal.

---
# RESUMEN FINAL

## ¿Por qué importa cada sección?

| Sección | Aprendiste | Impacto |
|---------|-----------|--------|
| 1 | Regularización ≠ Tuning | Tuning es crítico, no cosmético |
| 2 | 3 estrategias de búsqueda | Cómo elegir según contexto |
| 3 | Grid vs Random | Random es 3x más rápido, igual bueno |
| 4 | Bayesian Optimization | Inteligente, eficiente, Optuna fácil |
| 5 | Gotchas reales (3 trampas) | Evita fracasos en producción |
| 6 | Flujo y checklist | Cómo implementar correctamente |

## Lecciones finales

1. **Tuning = inversión** → toma tiempo, pero +10-20% métrica es típico
2. **Validación correcta > búsqueda agresiva** → hold-out test es mandatorio
3. **Bayesian Optimization es el futuro** → menos evaluaciones, mejor resultado
4. **Distribution shift es real** → monitorea en producción, no confíes solo en test
5. **Documenta decisiones** → facilita debug cuando falla en producción

---

**Siguiente paso**: Toma el código, adáptalo a TUS datos, y empieza con Grid Search en 2-3 parámetros clave.